
# Pure numpy/scipy derivation: does a "naive" complex-baseband noise convention
# match demodulated broadband noise?

**This notebook imports nothing from `si_qfi`, anywhere.** Every piece — the white
noise generator, the demodulator, and the "naive baseband" reference generator — is
implemented fresh, right here, using only `numpy` and `scipy.signal`. The goal is to
remove any possibility that a shared bug or shared helper inside `si_qfi` itself is
producing the ~4x discrepancy seen when comparing `si_qfi`'s own
`generate_baseband_noise()` against `generate_rf_noise()` + `demodulate()` — if the
same factor shows up here, using code that has never touched `si_qfi`, it isn't an
`si_qfi` implementation bug; it's a real property of the underlying signal processing.

Companion notebook `noise_psd_convention_verification.ipynb` runs the same checks
directly against `si_qfi`'s actual functions, for comparison.


In [1]:

import numpy as np
from scipy.signal import butter, filtfilt



## The three functions, written from scratch

1. **`gen_white_rf_noise`** — real white noise with an exactly-controlled one-sided
   PSD `S_v`, spanning the full Nyquist band. Same idea as any textbook "generate
   noise with a target PSD" recipe: draw complex Gaussian frequency-domain samples
   with the right per-bin variance, enforce Hermitian symmetry (so the inverse FFT is
   real), inverse-FFT.
2. **`demod_iq`** — the standard coherent I/Q demodulator: mix down by
   `2·exp(-i2πf_c t)`, low-pass filter. This "×2" is the standard single-sideband
   recovery factor in every communications textbook treatment of narrowband
   demodulation (see e.g. Proakis & Salehi, *Communication Systems Engineering*,
   or Van Trees, *Detection, Estimation, and Modulation Theory* — both derive this
   exact ×2-in-amplitude / 4x-in-power relationship for recovering the complex
   envelope of a real bandpass signal).
3. **`gen_naive_baseband_noise`** — the "naive" convention: draw complex baseband
   noise where the target `Var` is simply `S_v · fs` directly, with no bandpass-to-
   envelope conversion applied. This mirrors exactly what a generator would do if it
   treated the input `S_v` as already being the complex envelope's own PSD, rather
   than a physical one-sided PSD of a real bandpass source.


In [2]:

def gen_white_rf_noise(N, fs, S_v, rng):
    '''Real white noise, one-sided PSD = S_v (flat) over [0, fs/2].'''
    n_half = N // 2 + 1
    amp = np.sqrt(S_v * fs * N) / np.sqrt(2)
    Xk = amp * (rng.standard_normal(n_half) + 1j * rng.standard_normal(n_half))
    Xk[0] = Xk[0].real * np.sqrt(2)
    if N % 2 == 0:
        Xk[-1] = Xk[-1].real * np.sqrt(2)
    return np.fft.irfft(Xk, n=N)


def demod_iq(v_rf, t, f_c, B, filter_order=8):
    '''Standard coherent I/Q demodulation: mix by 2*exp(-i*2*pi*f_c*t), then
    low-pass filter at cutoff B. Returns the complex envelope I + jQ.'''
    fs = 1.0 / (t[1] - t[0])
    b, a = butter(N=filter_order, Wn=B / (fs / 2), btype="low")
    mix = 2.0 * v_rf * np.exp(-1j * 2 * np.pi * f_c * t)
    I = filtfilt(b, a, mix.real)
    Q = filtfilt(b, a, mix.imag)
    return I + 1j * Q


def gen_naive_baseband_noise(N, fs_bb, S_v, rng):
    '''"Naive" complex baseband noise: Var = S_v * fs_bb directly, no
    bandpass-to-envelope conversion -- matches treating S_v as if it were
    already the complex envelope's own two-sided PSD. The /sqrt(2) here
    matches si_qfi's noise/realization.py generate_baseband_noise() exactly
    (copied from its source, not reconstructed from memory) -- it's needed
    so that (randn + j*randn)/sqrt(2) has E[|.|^2]=1, giving E[|Xk|^2] =
    amp^2 = S_v*df per bin, not 2x that.'''
    df = fs_bb / N
    amp = np.sqrt(S_v * df)
    Xk = amp * (rng.standard_normal(N) + 1j * rng.standard_normal(N)) / np.sqrt(2)
    return np.fft.ifft(Xk) * N



## Step 1 — sanity-check each function's own advertised behavior, in isolation

Before comparing them to each other, confirm each one does what it claims on its own.


In [3]:

N = 400_000
fs = 40e9
S_v = 1e-15

rng = np.random.default_rng(100)
v_rf = gen_white_rf_noise(N, fs, S_v, rng)
print(f"gen_white_rf_noise:        Var = {np.var(v_rf):.4e}   target S_v*fs = {S_v*fs:.4e}   "
      f"ratio = {np.var(v_rf)/(S_v*fs):.4f}")
assert abs(np.var(v_rf)/(S_v*fs) - 1) < 0.05

rng = np.random.default_rng(101)
v_bb_naive = gen_naive_baseband_noise(N, fs, S_v, rng)
print(f"gen_naive_baseband_noise:  Var = {np.var(v_bb_naive):.4e}   target S_v*fs = {S_v*fs:.4e}   "
      f"ratio = {np.var(v_bb_naive)/(S_v*fs):.4f}")
assert abs(np.var(v_bb_naive)/(S_v*fs) - 1) < 0.05
print("\nBoth match their own Var = S_v * fs convention exactly, as designed.")


gen_white_rf_noise:        Var = 3.9908e-05   target S_v*fs = 4.0000e-05   ratio = 0.9977
gen_naive_baseband_noise:  Var = 3.9984e-05   target S_v*fs = 4.0000e-05   ratio = 0.9996

Both match their own Var = S_v * fs convention exactly, as designed.



## Step 2 — the actual comparison: same `S_v`, demodulated broadband noise vs. naive baseband

Draw broadband white noise with a known one-sided `S_v`, demodulate it down to a
`B`-wide-each-side complex envelope, and compare against `gen_naive_baseband_noise`
at a matching bandwidth (`fs_bb = 2B`) for the *same* `S_v`.


In [4]:

N = 2_000_000
fs_native = 40e9
f_c = 5e9
B = 2e9
S_v = 1e-15

rng = np.random.default_rng(200)
v_rf = gen_white_rf_noise(N, fs_native, S_v, rng)
t = np.arange(N) / fs_native
v_demod = demod_iq(v_rf, t, f_c, B)

edge = N // 10
var_demod = np.var(v_demod[edge:-edge])

fs_bb = 2 * B
rng2 = np.random.default_rng(201)
v_naive = gen_naive_baseband_noise(N, fs_bb, S_v, rng2)
var_naive = np.var(v_naive)

print(f"demodulated broadband noise:  Var = {var_demod:.4e}")
print(f"naive baseband convention:    Var = {var_naive:.4e}")
print(f"ratio (demodulated / naive):  {var_demod/var_naive:.3f}")
print()
print("predicted from first principles: 4.0  (see markdown derivation below)")
assert abs(var_demod/var_naive/4.0 - 1) < 0.15


demodulated broadband noise:  Var = 1.5101e-05
naive baseband convention:    Var = 3.9958e-06
ratio (demodulated / naive):  3.779

predicted from first principles: 4.0  (see markdown derivation below)



## Step 3 — first-principles derivation of the factor of 4

Real signal `v_rf(t)`, one-sided PSD `S_v` (flat), demodulated via
`v_bb(t) = LPF_B[2·v_rf(t)·exp(-i2πf_c t)]` — exactly what `demod_iq` above computes.

1. Mixing shifts the spectrum: `FT[2·v_rf·e^{-i2πf_c t}](f) = 2·V_rf(f+f_c)`.
2. Low-pass filtering to `|f|<B` keeps only `V_rf` evaluated over `[f_c-B, f_c+B]`
   — a one-sided slice of width `2B` — and nothing else (no other part of `v_rf`'s
   spectrum survives).
3. That slice carries power `P_slice = S_v · 2B` (flat one-sided PSD integrated over
   a `2B`-wide window).
4. Mixing (a unit-modulus multiply) doesn't change power; filtering only removes what
   it doesn't keep. So the *only* extra scaling is the explicit `2×` baked into the
   mixing step, applied in **amplitude** — which is `4×` in **power**:
   `Var(v_bb) = 4 · P_slice = 4 · S_v · 2B = 8 · S_v · B`.
5. Compare to the naive convention at matching bandwidth `fs_bb = 2B`:
   `Var_naive = S_v · fs_bb = S_v · 2B`.
6. Ratio: `(8·S_v·B) / (S_v·2B) = 4`.



## Step 4 — sanity check: narrowband content round-trips with NO factor of 4

If the factor of 4 were a bug in `demod_iq` itself (rather than a real property of
demodulating *broadband* content), it would also show up when demodulating a signal
that was already narrowband to begin with. It doesn't.


In [5]:

# 4a: deterministic narrowband pulse
N_bb = 2000
fs_bb = 4e9
fs_native = 40e9
f_c = 5e9

t_bb = np.arange(N_bb) / fs_bb
v_bb_known = np.exp(-((t_bb - t_bb[-1]/2)**2) / (2*(t_bb[-1]/6)**2)).astype(complex)

upsample = int(round(fs_native / fs_bb))
v_bb_up = np.repeat(v_bb_known, upsample)
t_up = np.arange(len(v_bb_up)) / fs_native
v_rf_det = np.real(v_bb_up * np.exp(1j * 2*np.pi*f_c*t_up))

v_recovered_det = demod_iq(v_rf_det, t_up, f_c, fs_bb/2)
print(f"deterministic pulse: original peak = {np.max(np.abs(v_bb_known)):.4f}, "
      f"recovered peak = {np.max(np.abs(v_recovered_det)):.4f}")


deterministic pulse: original peak = 1.0000, recovered peak = 1.0000


In [6]:

# 4b: genuinely narrowband NOISE (band-limited to +/-B BEFORE modulating up, unlike
# Step 2's broadband source)
N_up = 2_000_000
fs_native = 40e9
fs_bb = 4e9
B = fs_bb / 2
S_v_input = 1e-15
f_c = 5e9

freqs_full = np.fft.fftfreq(N_up, d=1/fs_native)
df_fine = fs_native / N_up
amp_fine = np.where(np.abs(freqs_full) < B, np.sqrt(S_v_input * df_fine), 0.0)
rng = np.random.default_rng(300)
# /sqrt(2) here for the same reason as gen_naive_baseband_noise above -- this is
# the same construction, just inlined with a partially-zero (band-limited) PSD
# array instead of a flat one.
Xk_fine = amp_fine * (rng.standard_normal(N_up) + 1j*rng.standard_normal(N_up)) / np.sqrt(2)
v_bb_narrowband = np.fft.ifft(Xk_fine) * N_up

var_start = np.var(v_bb_narrowband)
print(f"starting narrowband noise: Var = {var_start:.4e} (target S_v*fs_bb = {S_v_input*fs_bb:.4e})")

t_up = np.arange(N_up) / fs_native
v_rf_noise = np.real(v_bb_narrowband * np.exp(1j*2*np.pi*f_c*t_up))

v_recovered = demod_iq(v_rf_noise, t_up, f_c, B)
edge = N_up // 10
var_recovered = np.var(v_recovered[edge:-edge])
print(f"recovered (round-tripped): Var = {var_recovered:.4e}")
print(f"ratio recovered/original: {var_recovered/var_start:.3f}  (expect ~1.0, NOT ~4)")


starting narrowband noise: Var = 4.0002e-06 (target S_v*fs_bb = 4.0000e-06)


recovered (round-tripped): Var = 3.7250e-06
ratio recovered/original: 0.931  (expect ~1.0, NOT ~4)



## Step 5 — first-principles derivation: why does the narrowband round trip give
ratio ~1, with no factor of 4 at all?

Step 4 showed it empirically. This derives it, so the difference between Step 2/3
(broadband, ratio 4) and Step 4 (narrowband, ratio 1) stops looking like two
unrelated facts.

**Setup.** Start from a KNOWN complex baseband process `v_bb(t)`, band-limited to
`|f|<B`, with `Var(v_bb) = S_in · f_{s,bb}` (`f_{s,bb}=2B`) — exactly
`gen_naive_baseband_noise`'s own output. Modulate it up:

```
v_rf(t) = Re[ v_bb(t) · exp(i·2π·f_c·t) ]
        = (1/2)·v_bb(t)·exp(i2πf_c t) + (1/2)·conj(v_bb(t))·exp(-i2πf_c t)
```

**Step A — what does modulation do to the spectrum near `+f_c`?** Using the Fourier
shift property on each term:

```
V_rf(f) = (1/2)·V_bb(f - f_c)  +  (1/2)·conj(V_bb(-f - f_c))
```

For `f` near `+f_c` (write `f = f_c + δ`, `|δ|<B`): the first term is
`(1/2)·V_bb(δ)` — inside `v_bb`'s own support, so it's the "real" content. The second
term needs `V_bb(-2f_c - δ)`, and since `f_c ≫ B`, that argument is far outside
`v_bb`'s support — it's exactly zero. So near `f_c`:

```
V_rf(f) ≈ (1/2) · V_bb(f - f_c)
```

**Modulation already costs a factor of 1/2 in amplitude, 1/4 in power**, purely from
splitting `v_bb`'s power into the `Re[...]` construction. This has nothing to do with
demodulation yet — it's a property of how a complex baseband signal becomes a real
one at all.

**Step B — demodulate.** Mixing+LPF keeps only this `+f_c` image (the `-f_c` image
mixes to `-2f_c`, discarded). Its power, from Step A, is `(1/4)` of what `v_bb`
itself carries in that same slice. The standard `×2`-amplitude demodulation
correction multiplies power by `4`. Net factor: `(1/4) × 4 = 1` — **an exact
algebraic cancellation, for any `B`**, not a coincidence of this specific bandwidth.
Modulating up and immediately demodulating down are, by construction, exact inverses
of each other — of course you get back what you started with.

**Why the broadband case (Step 2/3) doesn't get this cancellation.** There, `S_v` is
`v_rf`'s *own* physical one-sided PSD, specified directly — it was never produced by
modulating some other narrowband `v_bb` up in the first place. There is no
"modulation ÷4" anywhere in that story to cancel against demodulation's `×4`. The
`×4` demodulation correctly extracts "how much of this wideband physical source's
power actually lands in my narrow band of interest" — a real, uncancelled effect,
because a real (not narrowband-derived) source was what went in.

**One-line summary:** demodulation always applies `×4` in power. For content that was
itself produced by modulating a narrowband complex envelope up, that `×4` exactly
cancels the `÷4` the modulation step introduced — net factor 1. For genuinely wideband
physical noise (never itself a modulated-up envelope), there's no such `÷4` to
cancel — net factor 4.


In [7]:

# Direct numerical check of Step A's claim: modulating v_bb up should cost EXACTLY
# a factor of 4 in power (1/2 in amplitude), independent of anything about demodulation.
N_up = 2_000_000
fs_native = 40e9
fs_bb = 4e9
S_v_check = 1e-15
f_c = 5e9

rng = np.random.default_rng(400)
v_bb_check = gen_naive_baseband_noise(N_up, fs_native, S_v_check, rng)  # flat over the WHOLE
                                                                          # native grid this time --
                                                                          # deliberately NOT band-limited,
                                                                          # to isolate the modulation
                                                                          # step's own effect cleanly
var_bb_check = np.var(v_bb_check)

t_up = np.arange(N_up) / fs_native
v_rf_check = np.real(v_bb_check * np.exp(1j*2*np.pi*f_c*t_up))
var_rf_check = np.var(v_rf_check)

print(f"Var(v_bb) before modulating up: {var_bb_check:.4e}")
print(f"Var(v_rf) after modulating up:  {var_rf_check:.4e}")
print(f"ratio (should be exactly 0.5, i.e. <v_rf^2> = (1/2)<|v_bb|^2>, "
      f"the standard real/complex power relation -- NOT the 1/4 from Step A, "
      f"because THIS ratio integrates power over BOTH the +fc and -fc images, "
      f"each carrying 1/4, summing to 1/2): {var_rf_check/var_bb_check:.4f}")
print()
print("Step A's '1/4 near +fc specifically' is exactly half of this 1/2 total --")
print("consistent: the +fc image alone carries half of v_rf's total power (the -fc")
print("image, its Hermitian mirror, carries the other half), i.e. 1/2 of 1/2 = 1/4.")


Var(v_bb) before modulating up: 3.9966e-05
Var(v_rf) after modulating up:  1.9978e-05
ratio (should be exactly 0.5, i.e. <v_rf^2> = (1/2)<|v_bb|^2>, the standard real/complex power relation -- NOT the 1/4 from Step A, because THIS ratio integrates power over BOTH the +fc and -fc images, each carrying 1/4, summing to 1/2): 0.4999

Step A's '1/4 near +fc specifically' is exactly half of this 1/2 total --
consistent: the +fc image alone carries half of v_rf's total power (the -fc
image, its Hermitian mirror, carries the other half), i.e. 1/2 of 1/2 = 1/4.



## Conclusion

Using code with **zero dependency on `si_qfi`** — a from-scratch white-noise
generator, a from-scratch coherent I/Q demodulator, and a from-scratch "naive
baseband" reference generator — demodulating broadband white noise gives **~4x** the
variance that the naive `Var = S_v · fs` convention computes directly for the same
`S_v`, matching a first-principles derivation exactly. The same demodulator recovers
narrowband content (deterministic or noise) with **no** such factor.

Step 5 explains why those aren't two separate facts: demodulation *always* applies a
`×4` power correction. Modulating a complex envelope up to a real signal *always*
costs a `÷4` first. For narrowband content that started as a modulated-up envelope
(Step 4), those two exactly cancel — algebraically, for any bandwidth, not by luck.
For genuinely wideband physical noise (Step 2/3) — which is what a real Johnson-noise
source actually is, and what `generate_rf_noise()` correctly represents — there is no
prior `÷4` to cancel, so the `×4` is real and uncancelled. `generate_baseband_noise()`
is missing it.
